# 2. ESM-2: generate variants from a seed sequence

- **Input:** The ubiquitin amino acid sequence.
- **Model information:** Sequence context and patterns learned from natural protein sequences. The model does not receive a structure.
- **Design objective:** Generate variants with a specified number of substitutions using the model's amino acid preferences.

Sample three variants, compare their substitutions, and predict the structure of a selected variant. Ubiquitin's experimental structure provides a reference for inspection.


*   Standard, most widely used protein language model
*   Originally developed by Facebook





*   Trained on a collection of AA sequences
*   Method: masked language modeling: input protein sequence, blank out ~10% of residues: predict filling in the blanks



## Setup
Select **Runtime → Change runtime type → T4 GPU**, then run the two cells below to load the tools and models.


In [1]:
!git -C BME305-protein-design-tools pull -q 2>/dev/null || git clone -q https://github.com/RomeroLab/BME305-protein-design-tools.git
%pip -q install --upgrade ./BME305-protein-design-tools
from protein_design_tools import load_pdb, load_ESMFold, load_ESM2, load_MPNN, load_RFdiffusion, predict_structure, MPNNdesign, ESM2design, RFdiffusion, alignment, view_structure, inspect_prediction, save_results


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.9/330.9 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 92.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 90.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
load_ESM2()
load_ESMFold();


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

Some weights of EsmForProteinFolding were not initialized from the model checkpoint at facebook/esmfold_v1 and are newly initialized: ['esm.contact_head.regression.bias', 'esm.contact_head.regression.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 1. Load the starting protein
Load the ubiquitin sequence used for generation and view its experimental structure as a reference. ESM-2 uses only the sequence in the next step.


In [3]:
native_seq, reference_pdb = load_pdb('1UBQ', chain='A')
view_structure(reference_pdb)


1UBQ, chain A: 76 residues


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 2. Generate variants
At each selected position, mask the residue and sample an amino acid from ESM-2's predictions. Each new sample becomes context for the next step. Higher temperature flattens the sampling distribution.


In [8]:
designs = ESM2design(native_seq, num_sequences=3, n_mutations=5, temperature=10, seed=7)


Generated 3 sequences with 5 substitutions each.


Chooses a number of random position to mask out, samples amino acids to replace it

## 3. Compare sequences
Compare the number and positions of substitutions. Examine changes in amino acid size, charge, or polarity. Mutation labels use positions starting at 1.


In [5]:
alignment({'ubiquitin':native_seq, **designs})



Positions 1–60
ubiquitin    MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYN
design_1     MQIFVKPLTSKTITLEVEPNDTIENVKAKIQDKEGIPPDQQALIFAGKQLVDGRTLSDYN
             ||||||.||.|||||||||.|||||||||||||||||||||.||||||||.|||||||||
design_2     MQIFVKTFTGKTDTLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAEKQLEDGRTLSDYN
             |||||||.||||.|||||||||||||||||||||||||||||||||.|||||||||||||
design_3     MQIFIKTLTGKEITLEVEPSDTIENVKRKIQDKEGIPPDQQRLIFAGKQLEDGRTPSDYN
             ||||.||||||.|||||||||||||||.|||||||||||||||||||||||||||.||||

Positions 61–76
ubiquitin    IQKESTLHLVLRLRGG
design_1     IQKESTLHLVLRLRGG
             ||||||||||||||||
design_2     IQKESTLHTVLRLRKG
             ||||||||.|||||.|
design_3     IQKEDTLHLVLRLRGG
             ||||.|||||||||||

design_1: 93% identity to ubiquitin; 5 differences
T7P, G10S, S20N, R42A, E51V

design_2: 93% identity to ubiquitin; 5 differences
L8F, I13D, G47E, L69T, G75K

design_3: 93% identity to ubiquitin; 5 differences
V5I, T12E, A28R, L56P, S65D

Restricted to the set number of differences n_mutations from model sequence

## 4. Predict and compare structures
Predict a selected variant with ESMFold and compare it with ubiquitin's experimental backbone. ESMFold uses the sequence alone, without the reference coordinates. RMSD measures backbone agreement; pLDDT describes confidence in the prediction. Inspect the overall fold and regions of disagreement alongside their confidence.


In [6]:
prediction_pdb = predict_structure(designs['design_1'], num_recycles=1)
metrics = inspect_prediction(reference_pdb, prediction_pdb)


Predicted 76 residues: /content/protein_demo/prediction_xxtzxl_b/prediction.pdb
Cα RMSD over all 76 residues: 0.91 Å
Mean Cα pLDDT: 90.1/100


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Gray: reference. Prediction confidence: blue ≥90, cyan 70–90, yellow 50–70, red <50.


## Discuss
- Does the predicted variant retain the overall fold? Are any changes near the sampled substitutions?
- How does sequence-only sampling differ from the backbone-conditioned ProteinMPNN demo?

Interesting: uncertainty in loops, whereas MPNN predicted structure was solid blue
*  Maybe MPNN was stabilizing loops more than WT Ubiquitin

Less likely than MPNN to change the activity of a protein because it's trained on sequences, which have conserved active site residues



## Save results
Download the designed sequences and structures for further analysis.


In [7]:
bundle = save_results(designs, reference_pdb, prediction_pdb)
from google.colab import files
files.download(str(bundle))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Reuse the tools
The same tools accept your own sequences and structures. See the [function reference](https://github.com/RomeroLab/BME305-protein-design-tools#function-reference) for use in later assignments.

## References
[ESM-2 and ESMFold](https://github.com/facebookresearch/esm) · [1UBQ](https://www.rcsb.org/structure/1UBQ)
